In [ ]:
from pathlib import Path
import os
from dotenv import load_dotenv

load_dotenv()

DATA_DIR      = Path("../data")
INFERENCE_URL = os.environ.get(
    "INFERENCE_URL",
    "http://localhost:8080/v1/models/csgo-match-predictor:predict",
)
N_SAMPLES = 10

print(f"Endpoint: {INFERENCE_URL}")

In [ ]:
import pandas as pd

X_test = pd.read_csv(DATA_DIR / "X_test.csv", index_col=0)
y_test = pd.read_csv(DATA_DIR / "y_test.csv", index_col=0).squeeze()

sample   = X_test.sample(N_SAMPLES, random_state=42)
expected = y_test.loc[sample.index].tolist()

print(f"Sampled {N_SAMPLES} rows from X_test")

In [ ]:
import requests

payload  = {"instances": sample.values.tolist()}
response = requests.post(INFERENCE_URL, json=payload, timeout=30)
response.raise_for_status()

predictions = response.json()["predictions"]
print(f"Status      : {response.status_code}")
print(f"Predictions : {predictions}")
print(f"Expected    : {expected}")

In [ ]:
assert len(predictions) == N_SAMPLES, f"Expected {N_SAMPLES} predictions, got {len(predictions)}"
assert all(p in [0, 1] for p in predictions), "Predictions must be 0 or 1"

correct = sum(p == e for p, e in zip(predictions, expected))
print(f"Correct: {correct}/{N_SAMPLES}")
print("Smoke test passed.")